# 🚗 Car Finder 2: Sort and Compare — PySpark RDD edition (model years 2000–2026)

This version preserves the original Car Finder 2 features: natural-language car search, six sorting choices, top-N results, two-car comparisons, comparison statistics, closest alternatives, interactive widgets, and CSV result downloads.

**RDD implementation:** Spark RDDs perform vocabulary discovery, filtering, sorting, aggregation, and comparison statistics. Pandas is used only to render small tables and export selected results in the notebook UI.

**How to use:** Run all cells in Google Colab, then search normally. Examples include `top 5 cheapest Toyota Corolla`, `lowest mileage Honda Civic 2015 or newer`, and `Corolla 2015 or newer vs Civic 2015 or newer`.


## 1. Setup and dataset
PySpark is installed if needed and a Spark session is started. Car records are loaded into a distributed RDD.


In [ ]:
import os, random, re, html
from dataclasses import dataclass, field
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Install PySpark in Google Colab if needed, then create the Spark session.
try:
    from pyspark.sql import SparkSession
except ImportError:
    !pip -q install pyspark
    from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("CarFinderSortCompareRDD").getOrCreate()
sc = spark.sparkContext
print("Spark version:", spark.version)


In [ ]:
def make_dataset():
    """Builds the synthetic car listings (same data every time because the random seed is fixed)."""
    random.seed(42)
    SCALE = 1
    Y0, Y1 = 2000, 2026

    # make -> [(model, body, new_price_usd, engine_cc, fuel_code, first_year, last_year)]
    # fuel_code: P petrol, D diesel, PD petrol/diesel, PH petrol or hybrid (hybrid only from 2010), E electric only
    CATALOG = {
        "Toyota": [("Corolla", "Sedan", 24000, 1800, "PH", 2000, 2026), ("Camry", "Sedan", 29000, 2500, "PH", 2000, 2026),
                   ("Yaris", "Hatchback", 18000, 1500, "P", 2000, 2026), ("Land Cruiser", "SUV", 75000, 4000, "PD", 2000, 2026),
                   ("Hilux", "Pickup", 35000, 2800, "D", 2000, 2026), ("Prius", "Hatchback", 28000, 1800, "H", 2003, 2026)],
        "Honda": [("Civic", "Sedan", 25000, 1800, "PH", 2000, 2026), ("City", "Sedan", 18000, 1500, "P", 2000, 2026),
                  ("Accord", "Sedan", 30000, 2000, "PH", 2000, 2026), ("CR-V", "SUV", 33000, 1500, "PH", 2000, 2026),
                  ("HR-V", "SUV", 26000, 1500, "P", 2015, 2026)],
        "Suzuki": [("Alto", "Hatchback", 8000, 660, "P", 2000, 2026), ("Cultus", "Hatchback", 11000, 1000, "P", 2000, 2017),
                   ("Swift", "Hatchback", 15000, 1200, "P", 2005, 2026), ("Wagon R", "Hatchback", 10000, 1000, "P", 2000, 2026),
                   ("Vitara", "SUV", 24000, 1600, "P", 2000, 2026)],
        "Hyundai": [("Elantra", "Sedan", 22000, 2000, "P", 2000, 2026), ("Tucson", "SUV", 30000, 2000, "PD", 2004, 2026),
                    ("Sonata", "Sedan", 27000, 2500, "PH", 2000, 2026), ("Ioniq 5", "SUV", 45000, 0, "E", 2021, 2026)],
        "Kia": [("Picanto", "Hatchback", 12000, 1000, "P", 2004, 2026), ("Sportage", "SUV", 29000, 2000, "PD", 2000, 2026),
                ("Sorento", "SUV", 35000, 2400, "PD", 2002, 2026), ("EV6", "SUV", 48000, 0, "E", 2021, 2026)],
        "Nissan": [("Sunny", "Sedan", 14000, 1500, "P", 2000, 2022), ("Patrol", "SUV", 60000, 4000, "PD", 2000, 2026),
                   ("Skyline", "Coupe", 38000, 3000, "P", 2000, 2019), ("Leaf", "Hatchback", 30000, 0, "E", 2011, 2026)],
        "Ford": [("Focus", "Hatchback", 20000, 1600, "P", 2000, 2018), ("Mustang", "Coupe", 40000, 5000, "P", 2000, 2026),
                 ("Ranger", "Pickup", 36000, 2200, "D", 2000, 2026), ("F-150", "Pickup", 45000, 3500, "P", 2000, 2026),
                 ("Explorer", "SUV", 42000, 3500, "P", 2000, 2026)],
        "BMW": [("3 Series", "Sedan", 45000, 2000, "PD", 2000, 2026), ("5 Series", "Sedan", 60000, 2000, "PD", 2000, 2026),
                ("X5", "SUV", 70000, 3000, "PD", 2000, 2026), ("Z4", "Convertible", 52000, 2000, "P", 2003, 2026)],
        "Mercedes-Benz": [("C-Class", "Sedan", 48000, 2000, "PD", 2000, 2026), ("E-Class", "Sedan", 62000, 2000, "PD", 2000, 2026),
                          ("S-Class", "Sedan", 115000, 3000, "PD", 2000, 2026), ("GLC", "SUV", 55000, 2000, "PD", 2015, 2026)],
        "Volkswagen": [("Golf", "Hatchback", 26000, 1400, "PD", 2000, 2026), ("Passat", "Sedan", 30000, 1800, "PD", 2000, 2022),
                       ("Tiguan", "SUV", 32000, 1400, "PD", 2008, 2026), ("Beetle", "Hatchback", 22000, 2000, "P", 2000, 2019)],
        "Mazda": [("Mazda 3", "Sedan", 24000, 2000, "P", 2003, 2026), ("CX-5", "SUV", 30000, 2500, "P", 2012, 2026),
                  ("MX-5", "Convertible", 31000, 2000, "P", 2000, 2026)],
        "Chevrolet": [("Camaro", "Coupe", 38000, 3600, "P", 2000, 2024), ("Corvette", "Coupe", 68000, 6200, "P", 2000, 2026),
                      ("Silverado", "Pickup", 45000, 5300, "P", 2000, 2026)],
        "Tesla": [("Model S", "Sedan", 95000, 0, "E", 2012, 2026), ("Model 3", "Sedan", 42000, 0, "E", 2017, 2026),
                  ("Model Y", "SUV", 50000, 0, "E", 2020, 2026), ("Model X", "SUV", 100000, 0, "E", 2016, 2026)],
    }
    COLORS = ["White", "Black", "Silver", "Red", "Blue", "Green", "Grey", "Yellow", "Maroon", "Gold"]
    COLOR_W = [20, 17, 17, 9, 11, 5, 12, 2, 4, 3]
    CITIES = ["Lahore", "Karachi", "Islamabad", "Rawalpindi", "Faisalabad", "Multan", "Peshawar"]


    def pick_fuel(code, year):
        if code == "E": return "Electric"
        if code == "H": return "Hybrid"
        if code == "PH": return "Hybrid" if year >= 2010 and random.random() < 0.3 else "Petrol"
        if code == "PD": return random.choice(["Petrol", "Diesel"])
        return "Diesel" if code == "D" else "Petrol"


    rows = []
    for make, models in CATALOG.items():
        for model, body, base, cc, code, first, last in models:
            lo, hi = max(first, Y0), min(last, Y1)
            for _ in range(random.randint(22, 38) * SCALE):
                year = random.randint(lo, hi)
                age = Y1 + 1 - year
                mileage = max(300, round(random.gauss(14000 * (age - 0.5), 8000), -2))
                price = base * (0.88 ** (age - 1)) * (1 - mileage / 1_500_000) * random.uniform(0.9, 1.1)
                price = max(price, base * 0.08)
                fuel = pick_fuel(code, year)
                rows.append({
                    "make": make, "model": model, "year": year,
                    "color": random.choices(COLORS, COLOR_W)[0],
                    "body_type": body, "fuel": fuel,
                    "transmission": "Automatic" if fuel == "Electric" else
                                    random.choices(["Manual", "Automatic"], [35, 65] if year > 2012 else [55, 45])[0],
                    "engine_cc": cc if fuel != "Electric" else 0,
                    "mileage_km": int(mileage), "price_usd": int(round(price, -1)),
                    "city": random.choice(CITIES),
                    "status": random.choices(["For Sale", "Sold", "Reserved"], [62, 30, 8])[0],
                })

    random.Random(1).shuffle(rows)
    for i, row in enumerate(rows, 1):
        row["car_id"] = f"CAR-{i:04d}"
    return rows


# If a file named cars_2000_2026.csv has been uploaded to Colab (folder icon on the left), it is used instead.
# It must have the columns: car_id, make, model, year, color, body_type, fuel, transmission,
# engine_cc, mileage_km, price_usd, city, status   (status = For Sale / Sold / Reserved)
REQUIRED = ["car_id", "make", "model", "year", "color", "body_type", "fuel", "transmission",
            "engine_cc", "mileage_km", "price_usd", "city", "status"]
CSV = "cars_2000_2026.csv"
if os.path.exists(CSV):
    loaded = spark.read.option("header", True).option("inferSchema", True).csv(CSV)
    if all(c in loaded.columns for c in REQUIRED):
        cars_rdd = loaded.select(*REQUIRED).rdd.map(lambda row: row.asDict(recursive=True))
        print(f"Loaded your file {CSV} with Spark.")
    else:
        print(f"{CSV} does not have the columns this notebook needs, so it was ignored.")
        cars_rdd = sc.parallelize(make_dataset())
else:
    cars_rdd = sc.parallelize(make_dataset())
    # Save generated data through Spark, without converting the dataset to pandas.
    spark.createDataFrame(cars_rdd).coalesce(1).write.mode("overwrite").option("header", True).csv("cars_2000_2026_spark_csv")
    print("Created the car dataset as an RDD. Spark CSV output is in cars_2000_2026_spark_csv/")

TEXT_COLS = ["make", "model", "color", "body_type", "fuel", "transmission", "city", "status"]
cars_rdd = cars_rdd.map(lambda r: {**r, **{c: str(r[c]).strip() for c in TEXT_COLS}}).cache()
total_cars = cars_rdd.count()
year_min_data = cars_rdd.map(lambda r: int(r["year"])).min()
year_max_data = cars_rdd.map(lambda r: int(r["year"])).max()
for_sale_count = cars_rdd.filter(lambda r: r["status"] == "For Sale").count()
make_count = cars_rdd.map(lambda r: r["make"]).distinct().count()
print(f"{total_cars:,} cars | model years {year_min_data}–{year_max_data} | {for_sale_count:,} for sale | {make_count} makes")
pd.DataFrame(cars_rdd.take(8))

## 2. The search engine (PySpark RDD)
The sentence parser keeps the same supported wording and criteria. Vocabulary is learned from the RDD. Search conditions use `RDD.filter`; sorting uses `RDD.sortBy`; alternatives also use RDD transformations.


In [ ]:
from __future__ import annotations
@dataclass
class Criteria:
    makes: list[str] = field(default_factory=list)
    models: list[str] = field(default_factory=list)
    colors: list[str] = field(default_factory=list)
    body_types: list[str] = field(default_factory=list)
    fuels: list[str] = field(default_factory=list)
    transmissions: list[str] = field(default_factory=list)
    cities: list[str] = field(default_factory=list)
    year_min: int | None = None
    year_max: int | None = None
    price_min: float | None = None
    price_max: float | None = None
    mileage_max: float | None = None
    unknown: list[str] = field(default_factory=list)   # words that match nothing in the dataset

    def active(self) -> dict:
        """Only the filters actually in use, as {label: value}."""
        out = {}
        for label, val in [("make", self.makes), ("model", self.models), ("color", self.colors),
                           ("body type", self.body_types), ("fuel", self.fuels),
                           ("transmission", self.transmissions), ("city", self.cities)]:
            if val:
                out[label] = " / ".join(val)
        if self.year_min is not None or self.year_max is not None:
            lo, hi = self.year_min, self.year_max
            out["year"] = str(lo) if lo == hi else f"{lo or '…'}–{hi or '…'}"
        if self.price_min is not None or self.price_max is not None:
            out["price"] = f"${self.price_min or 0:,.0f} – " + (f"${self.price_max:,.0f}" if self.price_max else "any")
        if self.mileage_max is not None:
            out["mileage"] = f"≤ {self.mileage_max:,.0f} km"
        if self.unknown:
            out["not in our listings"] = ", ".join(self.unknown)
        return out

    def without(self, label: str) -> "Criteria":
        c = Criteria(**self.__dict__)
        mapping = {"make": "makes", "model": "models", "color": "colors", "body type": "body_types",
                   "fuel": "fuels", "transmission": "transmissions", "city": "cities"}
        if label in mapping:
            setattr(c, mapping[label], [])
        elif label == "year":
            c.year_min = c.year_max = None
        elif label == "price":
            c.price_min = c.price_max = None
        elif label == "mileage":
            c.mileage_max = None
        elif label == "not in our listings":
            c.unknown = []
        return c


# ---------- parsing ----------
def _price_ctx(y: str, t: str) -> bool:
    """True if a 4-digit number like 2000 is clearly a price, not a model year."""
    return bool(
        re.search(r"(?:under|below|less than|cheaper than|max(?:imum)?|upto|up to|within|budget(?: of)?|over|above|"
                  r"more than|at least|min(?:imum)?|between)\s*\$?\s*" + y + r"(?!\d)", t)
        or re.search(r"\$\s?" + y, t) or re.search(y + r"\s?(?:usd|dollars)", t)
        or re.search(y + r"\s*(?:and|to|-)\s*\$?\s*\d{4,}", t))


def _flex(v: str) -> str:
    return re.escape(v.lower()).replace(r"\ ", r"[\s-]?").replace(r"\-", r"[\s-]?")


def _find(text: str, vocab) -> list[str]:
    found = []
    for v in sorted(set(vocab), key=len, reverse=True):          # longest first: "Land Cruiser" before "Land"
        pat = r"(?<![a-z0-9])" + _flex(v) + r"s?(?![a-z0-9])"
        if re.search(pat, text):
            found.append(v)
    return found


def _num(s: str) -> float:
    s = s.lower().replace(",", "").replace("$", "").strip()
    return float(s[:-1]) * 1000 if s.endswith("k") else float(s)


ALIASES = {"vw": "Volkswagen", "ev": "Electric", "evs": "Electric", "electrics": "Electric", "hybrids": "Hybrid",
           "benzin": "Petrol", "gasoline": "Petrol",
           "chevy": "Chevrolet", "landcruiser": "Land Cruiser", "gray": "Grey", "truck": "Pickup", "pickups": "Pickup"}
STOP = set("""a an the is are was be any for sale selling available availability car cars vehicle vehicles in on at with
under below over above than less more between and or to from after before since until till i we you my me want wanted
looking look need needed do does have has show find get buy new used old cheap cheaper best good nice please can could
km kms miles mile mi kilometers kilometres usd dollars dollar model year color colour of least max maximum min minimum
upto up budget within auto automatic manual stick transmission engine petrol diesel gas fuel city price mileage there
that this which it its one some around about approximately approx under ready available stock pickup suv sedan coupe hatchback
convertible wagon electric hybrid ev newer older later earlier onwards onward what whats where how much many sold""".split())

NUM = r"\$?\s?(\d[\d,]*(?:\.\d+)?\s?k?)"


def parse_query(text: str, vocab: dict) -> Criteria:
    t = " " + text.lower().strip() + " "
    t = re.sub(r"(?<![a-z])(?:mercedes[\s-]?benz|mercedes|benz|merc)(?![a-z])", "mercedes-benz", t)
    for alias, real in ALIASES.items():          # normalise nicknames before matching
        t = re.sub(r"(?<![a-z])" + alias + r"(?![a-z])", real.lower(), t)
    c = Criteria()
    c.models = _find(t, vocab["model"])
    c.makes = _find(t, vocab["make"])
    # a model already implies its make
    if c.models:
        implied = {mk for m in c.models for mk in vocab["model_makes"].get(m, ())}
        c.makes = [m for m in c.makes if m in implied]
    c.colors = _find(t, vocab["color"])
    c.body_types = _find(t, vocab["body_type"])
    c.fuels = _find(t, vocab["fuel"])
    c.cities = _find(t, vocab["city"])
    if re.search(r"\bauto(matic)?\b", t):
        c.transmissions = ["Automatic"]
    elif re.search(r"\bmanual\b|\bstick\b", t):
        c.transmissions = ["Manual"]

    # mileage first, so those numbers are not mistaken for price/year
    m = re.search(r"(?:under|below|less than|max|within|upto|up to)?\s*" + NUM + r"\s?(?:km|kms|kilometers|kilometres|miles|mi)\b", t)
    if m:
        c.mileage_max = _num(m.group(1)) * (1.609 if re.search(r"miles|mi\b", m.group(0)) else 1)
        t = t.replace(m.group(0), " ")

    # year range / single year (model years are 4-digit tokens from 2000 to 2026)
    years_rng = re.search(r"\b(20(?:0\d|1\d|2[0-6]))\s?(?:-|–|to|and|till|until)\s?(20(?:0\d|1\d|2[0-6]))\b", t)
    if years_rng:
        a, b = sorted((int(years_rng.group(1)), int(years_rng.group(2))))
        c.year_min, c.year_max = a, b
        t = t.replace(years_rng.group(0), " ")
    else:
        single = [y for y in re.findall(r"\b(20(?:0\d|1\d|2[0-6]))\b", t) if not _price_ctx(y, t)]
        if len(single) == 1:
            y = int(single[0])
            if re.search(r"(?:after|newer than|since|from)\s+" + single[0], t) or \
               re.search(single[0] + r"\s*(?:or|and)?\s*(?:newer|later|above|onwards?|\+)", t):
                c.year_min = y
            elif re.search(r"(?:before|older than|until|till)\s+" + single[0], t) or \
               re.search(single[0] + r"\s*(?:or|and)?\s*(?:older|earlier|below)", t):
                c.year_max = y
            else:
                c.year_min = c.year_max = y
            t = t.replace(single[0], " ", 1)
        elif len(single) >= 2:
            ys = sorted(int(x) for x in single)
            c.year_min, c.year_max = ys[0], ys[-1]
            for x in single:
                t = t.replace(x, " ", 1)

    # price
    pr = re.search(r"(?:between)\s*" + NUM + r"\s*(?:and|to|-)\s*" + NUM, t) or \
         re.search(NUM + r"\s*(?:-|to)\s*" + NUM, t)
    if pr and 'between' in pr.group(0) or (pr and "$" in pr.group(0)):
        lo, hi = sorted((_num(pr.group(1)), _num(pr.group(2))))
        c.price_min, c.price_max = lo, hi
    else:
        mx = re.search(r"(?:under|below|less than|cheaper than|max(?:imum)?|upto|up to|within|budget(?: of)?)\s*" + NUM, t)
        mn = re.search(r"(?:over|above|more than|at least|min(?:imum)?)\s*" + NUM, t)
        if mx:
            c.price_max = _num(mx.group(1))
        if mn:
            c.price_min = _num(mn.group(1))
        if not mx and not mn:
            d = re.search(r"\$\s?(\d[\d,]*k?)|(\d[\d,]*k?)\s?(?:usd|dollars)", t)
            if d:
                c.price_max = _num(d.group(1) or d.group(2))

    # anything left that we cannot place (e.g. a brand we do not list) must not be silently ignored
    left = t
    for v in (vocab["make"] + vocab["model"] + vocab["color"] + vocab["body_type"] + vocab["city"]):
        left = re.sub(r"(?<![a-z0-9])" + _flex(v) + r"s?(?![a-z0-9])", " ", left)
    words = re.findall(r"[a-z][a-z\-]{2,}", left)
    c.unknown = [w.title() for w in dict.fromkeys(words) if w not in STOP]
    return c

In [ ]:
VOCAB_COLS = ["make", "model", "color", "body_type", "fuel", "transmission", "city"]
FOR_SALE = "For Sale"

# Learn searchable terms from the RDD.
VOCAB = {
    col: sorted(cars_rdd.map(lambda r, col=col: r[col]).distinct().collect())
    for col in VOCAB_COLS
}
VOCAB["model_makes"] = (
    cars_rdd.map(lambda r: (r["model"], r["make"]))
            .distinct()
            .groupByKey()
            .mapValues(set)
            .collectAsMap()
)

SORTS = {
    "price_asc":    ("price_usd", True,  "cheapest first"),
    "price_desc":   ("price_usd", False, "most expensive first"),
    "mileage_asc":  ("mileage_km", True,  "lowest mileage first"),
    "mileage_desc": ("mileage_km", False, "highest mileage first"),
    "year_desc":    ("year", False, "newest first"),
    "year_asc":     ("year", True, "oldest first"),
}


def sort_rdd(rdd, key):
    col, ascending, _ = SORTS[key]
    if col == "price_usd":
        return rdd.sortBy(lambda r: float(r[col]) if ascending else -float(r[col]))
    # Preserve the original tie-break rule: equal mileage/year sorts by cheaper price.
    return rdd.sortBy(lambda r: (float(r[col]) if ascending else -float(r[col]),
                                 float(r["price_usd"])))


def apply_criteria(rdd, c):
    if c.unknown:
        return rdd.filter(lambda r: False)

    def matches(r):
        return all([
            not c.makes or r["make"] in c.makes,
            not c.models or r["model"] in c.models,
            not c.colors or r["color"] in c.colors,
            not c.body_types or r["body_type"] in c.body_types,
            not c.fuels or r["fuel"] in c.fuels,
            not c.transmissions or r["transmission"] in c.transmissions,
            not c.cities or r["city"] in c.cities,
            c.year_min is None or int(r["year"]) >= c.year_min,
            c.year_max is None or int(r["year"]) <= c.year_max,
            c.price_min is None or float(r["price_usd"]) >= c.price_min,
            c.price_max is None or float(r["price_usd"]) <= c.price_max,
            c.mileage_max is None or float(r["mileage_km"]) <= c.mileage_max,
        ])
    return rdd.filter(matches)


@dataclass
class Result:
    criteria: Criteria
    available: list
    unavailable: list
    alternatives: list

    @property
    def found(self):
        return len(self.available) > 0


def search(rdd, c, sort_key="price_asc"):
    matches = apply_criteria(rdd, c)
    available = sort_rdd(matches.filter(lambda r: r["status"] == FOR_SALE), sort_key).collect()
    unavailable = matches.filter(lambda r: r["status"] != FOR_SALE).collect()
    alternatives = []
    if not available:
        for label in c.active():
            alt = apply_criteria(rdd, c.without(label)).filter(lambda r: r["status"] == FOR_SALE)
            alt_rows = sort_rdd(alt, sort_key).collect()
            if alt_rows:
                alternatives.append((label, alt_rows))
        alternatives.sort(key=lambda x: len(x[1]))
    return Result(c, available, unavailable, alternatives)


## 3. Sorting words
Sorting words such as *cheapest*, *lowest mileage*, *newest* and *top 5* are removed from the query and converted into an RDD sort key and result limit.


In [ ]:
STOP.update({"list", "give", "tell", "first", "top", "compare", "versus", "vs", "side", "sorted", "sort", "order", "ordered", "by"})

SORT_PATTERNS = [
    ("price_desc",   r"\b(?:most expensive|priciest|highest[\s-]?priced?|price high to low|high to low)(?:\s+first)?\b"),
    ("price_asc",    r"\b(?:cheapest|least expensive|lowest[\s-]?priced?|lowest cost|sort(?:ed)? by price|price low to high|low to high)(?:\s+first)?\b"),
    ("mileage_asc",  r"\b(?:(?:lowest|least|low|fewest|less)[\s-]?(?:mileage|kms?|kilometers|kilometres)|least driven|lowest driven)(?:\s+first)?\b"),
    ("mileage_desc", r"\b(?:(?:highest|most|high)[\s-]?(?:mileage|kms?|kilometers|kilometres)|most driven)(?:\s+first)?\b"),
    ("year_desc",    r"\b(?:newest|latest|most recent|youngest)(?:\s+first)?\b"),
    ("year_asc",     r"\b(?:oldest|earliest)(?:\s+first)?\b"),
]


def extract_sort(text, default_key):
    # Returns (sort key, sentence without the sort words, how many to show or None, whether the user typed a sort word)
    found = []
    for key, pattern in SORT_PATTERNS:
        m = re.search(pattern, text, flags=re.I)
        if m:
            found.append((m.start(), key))
        text = re.sub(pattern, " ", text, flags=re.I)
    top = None
    m = re.search(r"\b(?:top|first)\s+(\d{1,3})\b", text, flags=re.I)
    if m:
        top = int(m.group(1))
        text = re.sub(r"\b(?:top|first)\s+\d{1,3}\b", " ", text, flags=re.I)
    text = re.sub(r"\s+", " ", text).strip()
    key = min(found)[1] if found else default_key          # the sort word that comes first in the sentence wins
    return key, text, top, bool(found)

## 4. Comparing two cars
Comparison requests search both sides independently. Summary metrics are computed using RDD `aggregate` and `reduceByKey`; only the compact summary values and a few displayed rows are brought to the driver.


In [ ]:
VS = r"\\s+(?:vs\\.?|versus|against)\\s+"
PREFIX = r"^\\s*(?:please\\s+)?(?:compare|comparison of|difference between)\\s+"


def split_compare(text):
    t = text.strip()
    body = re.sub(PREFIX, "", t, flags=re.I)
    had_prefix = body != t
    parts = [p.strip() for p in re.split(VS, body, flags=re.I) if p.strip()]
    if len(parts) >= 2:
        return parts
    if had_prefix:
        m = re.match(r"(?is)^(.+?)\\s+(?:and|with|to)\\s+(.+)$", body)
        return [m.group(1).strip(), m.group(2).strip()] if m else [body]
    return None


def _mode_rdd(rdd, column):
    counts = rdd.map(lambda r: (r[column], 1)).reduceByKey(lambda a, b: a + b)
    rows = counts.collect()
    return sorted(rows, key=lambda x: (-x[1], str(x[0])))[0][0] if rows else None


def side_stats(rdd):
    # RDD aggregate computes count, sums, minima and maxima without collecting car rows.
    zero = {"count": 0, "sum_price": 0.0, "min_price": float("inf"), "max_price": float("-inf"),
            "sum_km": 0.0, "min_km": float("inf"), "sum_year": 0.0,
            "new_year": -float("inf"), "old_year": float("inf")}
    def seq(a, r):
        p, km, y = float(r["price_usd"]), float(r["mileage_km"]), int(r["year"])
        a["count"] += 1
        a["sum_price"] += p
        a["min_price"] = min(a["min_price"], p)
        a["max_price"] = max(a["max_price"], p)
        a["sum_km"] += km
        a["min_km"] = min(a["min_km"], km)
        a["sum_year"] += y
        a["new_year"] = max(a["new_year"], y)
        a["old_year"] = min(a["old_year"], y)
        return a
    def comb(a, b):
        a["count"] += b["count"]
        for k in ["sum_price", "sum_km", "sum_year"]:
            a[k] += b[k]
        for k in ["min_price", "min_km", "old_year"]:
            a[k] = min(a[k], b[k])
        for k in ["max_price", "new_year"]:
            a[k] = max(a[k], b[k])
        return a
    agg = rdd.aggregate(zero, seq, comb)
    if agg["count"] == 0:
        return None
    return {"count": agg["count"], "min_price": agg["min_price"],
            "avg_price": agg["sum_price"] / agg["count"], "max_price": agg["max_price"],
            "avg_km": agg["sum_km"] / agg["count"], "min_km": agg["min_km"],
            "avg_year": agg["sum_year"] / agg["count"], "new_year": agg["new_year"],
            "old_year": agg["old_year"], "color": _mode_rdd(rdd, "color"),
            "fuel": _mode_rdd(rdd, "fuel"), "gearbox": _mode_rdd(rdd, "transmission")}


def compare_data(side_texts):
    sides = []
    for side in side_texts[:2]:
        rest = extract_sort(side, "price_asc")[1]
        c = parse_query(rest, VOCAB)
        result = search(cars_rdd, c) if c.active() else None
        stats = None
        if result is not None and result.found:
            side_rdd = apply_criteria(cars_rdd, c).filter(lambda r: r["status"] == FOR_SALE)
            stats = side_stats(side_rdd)
        sides.append({"name": side.strip(), "criteria": c, "result": result, "stats": stats})
    return sides


METRICS = [("Cars for sale", "count", "high", "int"), ("Cheapest price (USD)", "min_price", "low", "usd"),
           ("Average price (USD)", "avg_price", "low", "usd"), ("Most expensive (USD)", "max_price", None, "usd"),
           ("Average mileage (km)", "avg_km", "low", "num"), ("Lowest mileage (km)", "min_km", "low", "num"),
           ("Average model year", "avg_year", "high", "year1"), ("Newest model year", "new_year", "high", "text"),
           ("Oldest model year", "old_year", None, "text"), ("Most common colour", "color", None, "text"),
           ("Most common fuel", "fuel", None, "text"), ("Most common gearbox", "gearbox", None, "text")]


def show_value(v, kind):
    if v is None:
        return "—"
    if kind == "int":
        return f"{v:,}"
    if kind == "usd":
        return f"${v:,.0f}"
    if kind == "num":
        return f"{v:,.0f}"
    if kind == "year1":
        return f"{v:.1f}"
    return str(v)


## 5. The search box
The interactive widgets are retained. The underlying search, sorting, and comparison operations use PySpark RDDs.


In [ ]:
COLS = ["car_id", "make", "model", "year", "color", "body_type", "fuel", "transmission", "mileage_km", "price_usd", "city"]
LABELS = {"car_id": "ID", "make": "Make", "model": "Model", "year": "Year", "color": "Colour", "body_type": "Body",
          "fuel": "Fuel", "transmission": "Gearbox", "mileage_km": "Mileage (km)", "price_usd": "Price (USD)",
          "city": "City", "status": "Status"}
ROW_LIMIT = 500      # most rows shown at once
EXAMPLES = ["cheapest Toyota Corolla", "lowest mileage Honda Civic 2015 or newer", "top 5 newest electric Tesla",
            "Corolla 2015 or newer vs Civic 2015 or newer", "compare BMW 3 Series and Mercedes C-Class"]

text = widgets.Text(placeholder="e.g. cheapest silver Suzuki Swift in Lahore  |  Corolla vs Civic",
                    layout=widgets.Layout(width="68%"), continuous_update=False)
button = widgets.Button(description="Search", button_style="primary", icon="search", layout=widgets.Layout(width="130px"))
sort_menu = widgets.Dropdown(options=[(SORTS[k][2].capitalize(), k) for k in SORTS], value="price_asc",
                             description="Sort by:", layout=widgets.Layout(width="280px"))
out = widgets.Output()


def banner(message, ok):
    bg, fg, bar = ("#e8f5ee", "#14532d", "#16a34a") if ok else ("#fdecec", "#7f1d1d", "#dc2626")
    display(HTML(f'<div style="background:{bg};color:{fg};border-left:6px solid {bar};padding:12px 16px;'
                 f'border-radius:8px;font-size:17px;font-weight:600;margin:10px 0">{message}</div>'))


def chips(items):
    parts = []
    for k, v, kind in items:
        bg, fg = {"bad": ("#fee2e2", "#991b1b"), "sort": ("#dbeafe", "#1e3a8a")}.get(kind, ("#e2e8f0", "#1e293b"))
        parts.append(f'<span style="display:inline-block;background:{bg};color:{fg};padding:2px 10px;margin:0 6px 6px 0;'
                     f'border-radius:999px;font-size:13px">{html.escape(k)}: {html.escape(str(v))}</span>')
    display(HTML("".join(parts)))


def criteria_chips(active):
    return [(k, v, "bad" if k == "not in our listings" else "") for k, v in active.items()]


def show_table(rows, extra=(), limit=ROW_LIMIT):
    # Convert only the small slice being displayed into a pandas table.
    t = pd.DataFrame(rows[:limit]).reindex(columns=COLS + list(extra)).rename(columns=LABELS)
    display(t.style.hide(axis="index").format({"Mileage (km)": "{:,}", "Price (USD)": "{:,}"}))


def run_single(query, buttons=True):
    sort_key, rest, top, typed = extract_sort(query, sort_menu.value)
    c = parse_query(rest, VOCAB)
    active = c.active()
    if not active:
        display(HTML("<p>Add a make, model, colour, year or price so we can search. Example: <i>cheapest white Honda Accord 2012</i>.</p>"))
        return
    r = search(cars_rdd, c, sort_key)
    items = criteria_chips(active) + [("sorted", SORTS[sort_key][2], "sort")]
    if top:
        items.append(("show", f"top {top}", "sort"))
    chips(items)
    if r.found:
        n = len(r.available)
        shown = r.available[:top] if top else r.available[:ROW_LIMIT]
        banner(f"Yes, {n:,} matching car{'s' if n != 1 else ''} for sale.", True)
        if top and n > top:
            display(HTML(f"<p>Showing the top {top} of {n:,} matches ({SORTS[sort_key][2]}).</p>"))
        elif n > ROW_LIMIT:
            display(HTML(f"<p>Showing the first {ROW_LIMIT} of {n:,} matches ({SORTS[sort_key][2]}).</p>"))
        show_table(shown)
        if buttons:
            dl = widgets.Button(description="Download results (CSV)", icon="download")

            def download(_, rows=shown):
                with out:
                    try:
                        from google.colab import files
                        pd.DataFrame(rows).reindex(columns=COLS).to_csv("results.csv", index=False)
                        files.download("results.csv")
                    except ImportError:
                        print("The download button works inside Google Colab.")
            dl.on_click(download)
            display(dl)
        return
    banner("No, nothing matching is for sale right now.", False)
    if c.unknown:
        display(HTML(f"<p>We have no listings for <b>{html.escape(', '.join(c.unknown))}</b>. "
                     "Check the spelling, or ask for a different make or model.</p>"))
    if len(r.unavailable):
        display(HTML(f"<p>{len(r.unavailable)} matching car(s) exist but are already sold or reserved:</p>"))
        show_table(r.unavailable, extra=["status"], limit=50)
    if r.alternatives:
        display(HTML("<h3>Closest alternatives</h3>"))
        for label, alt in r.alternatives[:3]:
            what = "Ignoring the unknown word" if label == "not in our listings" else f"If you are flexible on {label}"
            display(HTML(f"<p><b>{what}</b>: {len(alt):,} car(s) for sale</p>"))
            show_table(alt, limit=8)


TD = "padding:7px 12px;border:1px solid #cbd5e1;"
BEST = "background:#e8f5ee;color:#14532d;font-weight:700;"


def run_compare(side_texts):
    if len(side_texts) < 2:
        display(HTML("<p>To compare, give two cars. Example: <i>Toyota Corolla 2018 vs Honda Civic 2018</i>.</p>"))
        return
    if len(side_texts) > 2:
        display(HTML("<p>Comparison works with two cars at a time, so only the first two are used.</p>"))
    sides = compare_data(side_texts)
    a, b = sides
    rows = ["<tr><th style='%s;text-align:left'></th>%s</tr>" % (TD, "".join(
        f"<th style='{TD}text-align:right'>{html.escape(s['name'])}</th>" for s in sides))]
    rows.append(f"<tr><td style='{TD}color:#64748b'>We searched for</td>" + "".join(
        f"<td style='{TD}text-align:right;color:#64748b'>{html.escape(', '.join(f'{k}: {v}' for k, v in s['criteria'].active().items()))}</td>"
        for s in sides) + "</tr>")
    for label, key, better, kind in METRICS:
        va = a["stats"][key] if a["stats"] else None
        vb = b["stats"][key] if b["stats"] else None
        win = None
        if better and va is not None and vb is not None and va != vb:
            win = 0 if (va < vb) == (better == "low") else 1
        cells = "".join(f"<td style='{TD}text-align:right;{BEST if win == i else ''}'>{show_value(v, kind)}</td>"
                        for i, v in enumerate((va, vb)))
        rows.append(f"<tr><td style='{TD}font-weight:600'>{label}</td>{cells}</tr>")
    display(HTML("<table style='border-collapse:collapse;margin:10px 0;font-size:14px'>" + "".join(rows) + "</table>"
                 "<div style='color:#64748b;font-size:12px'>Green marks the better value in each row.</div>"))

    notes = []
    if a["stats"] and b["stats"]:
        sa, sb = a["stats"], b["stats"]
        name = lambda i: f"<b>{html.escape(sides[i]['name'])}</b>"
        if sa["avg_price"] != sb["avg_price"]:
            i = 0 if sa["avg_price"] < sb["avg_price"] else 1
            hi, lo = max(sa["avg_price"], sb["avg_price"]), min(sa["avg_price"], sb["avg_price"])
            notes.append(f"Price: {name(i)} is cheaper on average, by ${hi - lo:,.0f} ({(hi - lo) / hi:.0%}).")
        if sa["avg_km"] != sb["avg_km"]:
            i = 0 if sa["avg_km"] < sb["avg_km"] else 1
            notes.append(f"Mileage: {name(i)} has lower mileage on average, by {abs(sa['avg_km'] - sb['avg_km']):,.0f} km.")
        if sa["avg_year"] != sb["avg_year"]:
            i = 0 if sa["avg_year"] > sb["avg_year"] else 1
            notes.append(f"Age: {name(i)} is newer on average, by {abs(sa['avg_year'] - sb['avg_year']):.1f} years.")
        if sa["count"] != sb["count"]:
            i = 0 if sa["count"] > sb["count"] else 1
            notes.append(f"Choice: {name(i)} has more cars for sale ({max(sa['count'], sb['count']):,} against {min(sa['count'], sb['count']):,}).")
        notes.append("Which one is better depends on whether price, mileage or age matters most to you.")
        banner("Both sides have cars for sale. Here is how they compare.", True)
        display(HTML("<ul>" + "".join(f"<li>{n}</li>" for n in notes) + "</ul>"))
    for s in sides:
        if s["stats"] is None:
            msg = f"Nothing for sale matches <b>{html.escape(s['name'])}</b>, so there is nothing to compare."
            if s["criteria"].unknown:
                msg += f" We have no listings for {html.escape(', '.join(s['criteria'].unknown))}."
            elif s["result"] is not None and s["result"].alternatives:
                label, alt = s["result"].alternatives[0]
                msg += f" Closest alternative: if you are flexible on {html.escape(label)}, {len(alt):,} car(s) are for sale."
            banner(msg, False)
    for s in sides:
        if s["stats"] is not None:
            display(HTML(f"<h4 style='margin-bottom:2px'>Cheapest cars: {html.escape(s['name'])}</h4>"))
            show_table(s["result"].available, limit=3)


def show_answer(query, buttons=True):
    query = query.strip()
    if not query:
        return
    sides = split_compare(query)
    if sides is not None:
        run_compare(sides)
    else:
        run_single(query, buttons)


def run_search(query):
    with out:
        clear_output(wait=True)
        show_answer(query)


def on_search(_):
    run_search(text.value)


def on_sort_change(_):
    if text.value.strip():
        run_search(text.value)


def use_example(b):
    text.value = b.description
    run_search(b.description)


button.on_click(on_search)
text.observe(on_search, names="value")          # pressing Enter in the box also searches
sort_menu.observe(on_sort_change, names="value")
example_buttons = []
for e in EXAMPLES:
    b = widgets.Button(description=e, layout=widgets.Layout(width="auto"))
    b.on_click(use_example)
    example_buttons.append(b)

display(HTML("<h3 style='margin-bottom:4px'>Car Finder 2: Sort and Compare</h3><div style='color:#64748b;margin-bottom:8px'>"
             "Is it for sale? Sort the results, or put <b>vs</b> between two cars to compare them. Model years 2000–2026.</div>"))
display(widgets.HBox([text, button]))
display(sort_menu)
display(HTML("<div style='color:#64748b;margin-top:8px'>Try:</div>"))
display(widgets.HBox(example_buttons, layout=widgets.Layout(flex_flow="row wrap")))
display(out)

## 6. Sample results
These sample searches use the same RDD-based engine.


In [ ]:
for q in ["top 5 cheapest Toyota Corolla", "lowest mileage Honda Civic 2015 or newer", "top 5 newest electric Tesla",
          "Corolla 2015 or newer vs Civic 2015 or newer", "compare BMW 3 Series and Mercedes C-Class", "blue Ferrari 2010"]:
    display(HTML(f"<h4 style='margin:18px 0 0 0'>Search: {html.escape(q)}</h4>"))
    show_answer(q, buttons=False)

## 7. Limitations
- **The data is synthetic** unless you upload `cars_2000_2026.csv` with the required columns.
- **Rule-based sentence parsing** is retained: no spelling correction or negation handling.
- **All conditions must match together.** Closest alternatives drop one condition at a time.
- **Four-digit numbers can mean a year or price**, depending on phrasing.
- **Prices are USD** and the parser supports English phrasing.
- **RDD operations are distributed.** Search result rows are collected to the driver for the interactive display/download. Comparison metrics are aggregated on the RDD.
- **Colab sessions are temporary.** Run all cells again after a runtime restart.
